In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path.cwd()

chunks = []

for chunk in pd.read_csv(
    data_dir / "la.data.64.County.txt",
    sep="\t",
    dtype=str,
    chunksize=100_000
):
    chunk.columns = chunk.columns.str.strip()

    chunk["year"] = chunk["year"].str.strip()
    chunk["period"] = chunk["period"].str.strip()

    filtered = chunk[
        chunk["year"].isin(
            ["2019", "2020", "2021", "2022", "2023", "2024"]
        )
        & (chunk["period"] == "M13")
    ].copy()

    chunks.append(filtered)

bls = pd.concat(chunks, ignore_index=True)

for col in bls.select_dtypes(include="object").columns:
    bls[col] = bls[col].str.strip()

print("Total rows:", len(bls))
print("\nColumns:", bls.columns.tolist())
print("\nFirst five rows:")
print(bls.head())
print("\nRecords by year:")
print(bls["year"].value_counts().sort_index())

Total rows: 77300

Columns: ['series_id', 'year', 'period', 'value', 'footnote_codes']

First five rows:
              series_id  year period value footnote_codes
0  LAUCN010010000000003  2019    M13   2.9            NaN
1  LAUCN010010000000003  2020    M13   5.3            NaN
2  LAUCN010010000000003  2021    M13   2.8            NaN
3  LAUCN010010000000003  2022    M13   2.2            NaN
4  LAUCN010010000000003  2023    M13   2.1            NaN

Records by year:
year
2019    12880
2020    12884
2021    12884
2022    12884
2023    12884
2024    12884
Name: count, dtype: int64


In [ ]:
bls["county_fips"] = bls["series_id"].str[5:10]

bls["measure_code"] = bls["series_id"].str[-2:]

print(
    bls[
        ["series_id", "county_fips", "measure_code", "year", "value"]
    ].head(10)
)

print("\nMeasurement counts:")
print(bls["measure_code"].value_counts())


              series_id county_fips measure_code  year value
0  LAUCN010010000000003       01001           03  2019   2.9
1  LAUCN010010000000003       01001           03  2020   5.3
2  LAUCN010010000000003       01001           03  2021   2.8
3  LAUCN010010000000003       01001           03  2022   2.2
4  LAUCN010010000000003       01001           03  2023   2.1
5  LAUCN010010000000003       01001           03  2024   2.6
6  LAUCN010010000000004       01001           04  2019   778
7  LAUCN010010000000004       01001           04  2020  1400
8  LAUCN010010000000004       01001           04  2021   739
9  LAUCN010010000000004       01001           04  2022   599

Measurement counts:
measure_code
03    19325
04    19325
05    19325
06    19325
Name: count, dtype: int64


In [ ]:

measure_names = {
    "03": "unemployment_rate",
    "04": "unemployment",
    "05": "employment",
    "06": "labor_force"
}

bls["measure"] = bls["measure_code"].map(measure_names)

bls["value"] = pd.to_numeric(bls["value"], errors="coerce")

bls_clean = bls.pivot(
    index=["county_fips", "year"],
    columns="measure",
    values="value"
).reset_index()

bls_clean.columns.name = None

bls_clean = bls_clean[
    [
        "county_fips",
        "year",
        "labor_force",
        "employment",
        "unemployment",
        "unemployment_rate"
    ]
]

print("Total county-year records:", len(bls_clean))
print(bls_clean.head())

Total county-year records: 19325
  county_fips  year  labor_force  employment  unemployment  unemployment_rate
0       01001  2019      26639.0     25861.0         778.0                2.9
1       01001  2020      26458.0     25058.0        1400.0                5.3
2       01001  2021      26742.0     26003.0         739.0                2.8
3       01001  2022      27128.0     26529.0         599.0                2.2
4       01001  2023      27823.0     27227.0         596.0                2.1


In [10]:
print("Missing values:")
print(bls_clean.isna().sum())

print("\nDuplicate county-year records:")
print(bls_clean.duplicated(["county_fips", "year"]).sum())

print("\nRecords by year:")
print(bls_clean["year"].value_counts().sort_index())

Missing values:
county_fips           0
year                  0
labor_force          78
employment           78
unemployment         78
unemployment_rate    78
dtype: int64

Duplicate county-year records:
0

Records by year:
year
2019    3220
2020    3221
2021    3221
2022    3221
2023    3221
2024    3221
Name: count, dtype: int64


In [ ]:
missing_bls = bls_clean[
    bls_clean["unemployment_rate"].isna()
]

print("Records with missing unemployment rates:", len(missing_bls))

print("\nMissing records by year:")
print(missing_bls["year"].value_counts().sort_index())

print("\nSample missing records:")
print(missing_bls.head(15))

measure_cols = [
    "labor_force",
    "employment",
    "unemployment",
    "unemployment_rate"
]

print("\nRecords missing all four measurements:")
print(bls_clean[measure_cols].isna().all(axis=1).sum())

Records with missing unemployment rates: 78

Missing records by year:
year
2020    78
Name: count, dtype: int64

Sample missing records:
      county_fips  year  labor_force  employment  unemployment  \
18858       72001  2020          NaN         NaN           NaN   
18864       72003  2020          NaN         NaN           NaN   
18870       72005  2020          NaN         NaN           NaN   
18876       72007  2020          NaN         NaN           NaN   
18882       72009  2020          NaN         NaN           NaN   
18888       72011  2020          NaN         NaN           NaN   
18894       72013  2020          NaN         NaN           NaN   
18900       72015  2020          NaN         NaN           NaN   
18906       72017  2020          NaN         NaN           NaN   
18912       72019  2020          NaN         NaN           NaN   
18918       72021  2020          NaN         NaN           NaN   
18924       72023  2020          NaN         NaN           NaN   
18930

In [ ]:
valid_bls = bls_clean.dropna(
    subset=[
        "labor_force",
        "employment",
        "unemployment",
        "unemployment_rate"
    ]
).copy()

valid_bls["labor_force_difference"] = (
    valid_bls["labor_force"]
    - valid_bls["employment"]
    - valid_bls["unemployment"]
)

print("Labor force differences:")
print(valid_bls["labor_force_difference"].describe())

valid_bls["calculated_rate"] = (
    valid_bls["unemployment"]
    / valid_bls["labor_force"]
    * 100
)

valid_bls["rate_difference"] = (
    valid_bls["unemployment_rate"]
    - valid_bls["calculated_rate"]
).abs()

print("\nUnemployment rate differences:")
print(valid_bls["rate_difference"].describe())

Labor force differences:
count    19247.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: labor_force_difference, dtype: float64

Unemployment rate differences:
count    19247.000000
mean         0.025153
std          0.014434
min          0.000000
25%          0.012735
50%          0.025335
75%          0.037574
max          0.050000
Name: rate_difference, dtype: float64


In [ ]:

bls_final = bls_clean[
    ~bls_clean["county_fips"].str.startswith("72")
].copy()

print("Original records:", len(bls_clean))
print("Records after excluding Puerto Rico:", len(bls_final))

print("\nMissing values:")
print(bls_final.isna().sum())

print("\nDuplicate county-year records:")
print(bls_final.duplicated(["county_fips", "year"]).sum())

print("\nRecords by year:")
print(bls_final["year"].value_counts().sort_index())

output_dir = Path.cwd() / "processed"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "bls_county_unemployment_2019_2024.csv"

bls_final.to_csv(output_file, index=False)

print("\nSaved to:", output_file)

Original records: 19325
Records after excluding Puerto Rico: 18857

Missing values:
county_fips          0
year                 0
labor_force          0
employment           0
unemployment         0
unemployment_rate    0
dtype: int64

Duplicate county-year records:
0

Records by year:
year
2019    3142
2020    3143
2021    3143
2022    3143
2023    3143
2024    3143
Name: count, dtype: int64

Saved to: c:\Users\sszat\Desktop\SCHOOL STUFF\College Doc Stuff\IS 477\IS 477 Project\Datasets\Bureau of Labor Statistics local unemployment statistics\processed\bls_county_unemployment_2019_2024.csv
